# Semana 4 — A\* cambiando diez líneas

**Plan de clase:** [`../weeks/week-04.md`](../weeks/week-04.md) · **Diapositivas:** [`../slides/week-04/deck.md`](../slides/week-04/deck.md)

Programación en vivo, 25 minutos. Reutiliza el código de la semana 3 y cambia el orden
de la frontera de `g` a `g + h`: el comportamiento del algoritmo cambia por completo.

**Este notebook produce una tabla.** Esa tabla resume toda la clase,
y su última fila conecta con cada comparación con LLM del curso.


## 1 · El código de la semana 3, sin cambios

Normalmente usarías `from aicourse.search import ...`. Aquí se repite para que el
notebook sea autónomo.


In [1]:
# heapq gestiona la cola de prioridad; time permite medir la duración de cada búsqueda.
# Los imports se mantienen como en el notebook original.
import heapq, itertools, time

# El tablero 3×3 se representa como una tupla de nueve enteros, por filas.
# El 0 representa el espacio vacío; una tupla de enteros puede ser clave de un diccionario.
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)


# POO: una clase define los datos y las operaciones de los objetos que se crean a partir de ella.
# Cada objeto Node representa un camino hasta un estado: guarda el tablero, su padre,
# la acción que permitió llegar y el costo acumulado g. Dos nodos pueden tener el mismo
# tablero y corresponder a caminos diferentes. Node y EightPuzzle no heredan entre sí.
class Node:
    # __slots__ declara los únicos atributos de estas instancias y reduce su consumo de memoria.
    # No vuelve inmutable al nodo: state, parent, action y g todavía pueden cambiar.
    __slots__ = ("state", "parent", "action", "g")
    # Node(...) crea una instancia y llama a __init__ para inicializarla.
    # self es esa instancia; Python lo pasa automáticamente al invocar sus métodos.
    # parent referencia otro objeto Node: es composición de objetos, no herencia.
    # En el nodo raíz, parent y action son None y el costo inicial es cero.
    def __init__(self, state, parent=None, action=None, g=0):
        self.state, self.parent, self.action, self.g = state, parent, action, g
    # node.path() es una llamada a un método de instancia: node se recibe como self.
    # Devuelve la lista de acciones del camino, no una lista de tableros.
    def path(self):
        # La variable local node empieza apuntando al nodo final y recorre sus padres.
        # Cambiar esta referencia local no modifica los nodos ni sus enlaces parent.
        node, out = self, []
        while node.parent is not None:
            out.append(node.action); node = node.parent
        # Las acciones se recogieron desde el final hasta el inicio; el corte [::-1] invierte la lista.
        return out[::-1]


# EightPuzzle describe el problema; Node registra los caminos que explora el algoritmo.
# La búsqueda usa la interfaz actions/result/is_goal/step_cost del objeto problema.
# Esto permite reutilizar el algoritmo con otro dominio que ofrezca esa misma interfaz,
# sin necesitar aquí una clase base común ni una relación de herencia explícita.
class EightPuzzle:
    # MOVES es un atributo de CLASE compartido por las instancias, consultado mediante self.MOVES.
    # Cada clave es una posición del espacio vacío y sus valores son posiciones vecinas.
    # Por ejemplo, desde el índice 0 puede intercambiarse con el índice 1 o con el 3.
    MOVES = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5),
             3: (0, 4, 6), 4: (1, 3, 5, 7), 5: (2, 4, 8),
             6: (3, 7), 7: (4, 6, 8), 8: (5, 7)}
    # p = EightPuzzle(start) crea un objeto con sus propios atributos initial y goal.
    # self es el objeto problema, mientras que state será el tablero concreto que evalúa cada método.
    def __init__(self, initial, goal=GOAL):
        self.initial, self.goal = initial, goal
    # state.index(0) localiza el vacío. La acción devuelta es el índice de una casilla
    # vecina con la que intercambiarlo, no una letra de dirección ni el valor de la ficha.
    def actions(self, state):
        return self.MOVES[state.index(0)]
    # La tupla original no se modifica: se copia a una lista, se intercambian dos posiciones
    # y se devuelve otra tupla. Así los estados ya usados como claves permanecen estables.
    def result(self, state, action):
        s = list(state); b = state.index(0)
        s[b], s[action] = s[action], s[b]
        return tuple(s)
    # Compara el contenido de las tuplas, no la identidad de los objetos.
    def is_goal(self, state):
        return state == self.goal
    # Cada movimiento cuesta 1: en este problema, el costo g coincide con el número de pasos.
    def step_cost(self, state, action):
        return 1


# profundidades óptimas verificadas (véase el apéndice de la semana 3)
D8  = (0, 4, 2, 5, 1, 3, 7, 8, 6)
D12 = (5, 4, 2, 7, 0, 3, 8, 1, 6)
D16 = (7, 5, 2, 4, 0, 3, 8, 1, 6)
print("week 3 machinery loaded")

week 3 machinery loaded


## 2 · A\* — Los cambios

Compara este código con `search` de la semana 3. Las diferencias: la frontera se ordena por
`g + h(state)` en lugar de `g`, y mantenemos un diccionario `best_g` para que una ruta más barata
hacia un estado ya visitado pueda reemplazar a una más costosa.


In [2]:
# problem es el objeto que define el dominio. h es una función que estima el costo
# restante a partir de un estado. Aquí se llama h(state), sin pasar problem como argumento.
# El resultado es una tupla: objeto Node objetivo o None, y número de expansiones.
def astar(problem, h):
    # Instanciación del nodo raíz a partir del atributo initial del objeto problema.
    start = Node(problem.initial)
    # La frontera es una lista usada como montículo mínimo mediante heapq.
    # Sus tuplas se ordenan por f=g+h y después por un contador único de desempate.
    # En el inicio g=0, por eso la prioridad inicial solo contiene h(start.state).
    frontier = [(h(start.state), 0, start)]      # (f, desempate, nodo)
    # Diccionario estado → menor costo conocido. Se compara el contenido del estado,
    # no la identidad del Node, para reconocer rutas alternativas al mismo tablero.
    best_g = {problem.initial: 0}
    # expansions mide trabajo de búsqueda; tiebreak sirve solo para ordenar empates.
    expansions, tiebreak = 0, 0

    while frontier:
        # Retira el nodo con menor prioridad. _ recibe el contador, que aquí no necesitamos usar.
        f, _, node = heapq.heappop(frontier)
        # Comprueba el objetivo al retirar el nodo, no apenas lo genera.
        # El nodo objetivo no se cuenta como una expansión porque no se generan sus hijos.
        if problem.is_goal(node.state):
            return node, expansions
        # Puede quedar una entrada antigua en la frontera tras encontrar un camino más barato.
        # get consulta el mejor costo y usa infinito si no existe; una entrada peor se descarta.
        if node.g > best_g.get(node.state, float("inf")):
            continue                              # entrada obsoleta: ya existe un camino de menor costo
        # Solo cuenta nodos no descartados cuyos sucesores se van a generar.
        expansions += 1
        for action in problem.actions(node.state):
            # Llama a un método del objeto problema para calcular el estado sucesor.
            s2 = problem.result(node.state, action)
            # g2 es el costo REAL acumulado hasta el hijo; todavía no incluye la heurística.
            g2 = node.g + problem.step_cost(node.state, action)
            # Solo conserva una mejora estricta; también permite volver a explorar un estado
            # si se descubre una ruta mejor. No hay un conjunto cerrado que lo impida.
            if g2 < best_g.get(s2, float("inf")):
                best_g[s2] = g2
                # El contador único evita que heapq intente comparar directamente dos objetos Node,
                # que no tienen un método __lt__ para definir cuál es menor.
                tiebreak += 1
                # f=g2+h(s2) es una estimación; el Node hijo guarda g2 sin sumar h.
                # El enlace con node permite reconstruir después todo el plan mediante path().
                heapq.heappush(frontier,
                               (g2 + h(s2), tiebreak, Node(s2, node, action, g2)))
    # Si no quedan nodos por explorar y no se encontró objetivo, informa que no hay solución encontrada.
    return None, expansions

## 3 · Cuatro heurísticas

Dos admisibles, una trivialmente admisible y una deliberadamente **no** admisible.


In [3]:
def h_zero(s):
    """Admisible, pero no informa. A* se convierte en búsqueda de costo uniforme."""
    # No estima ningún costo restante: con h=0, la prioridad depende solo del costo acumulado g.
    return 0


def h_misplaced(s):
    """Cuenta las fichas mal colocadas. Admisible: cada una necesita al menos un movimiento."""
    # enumerate produce pares (índice, valor). Cuenta fichas fuera de su posición e ignora el vacío.
    # Una ficha mal colocada necesita al menos un movimiento para llegar a su lugar.
    return sum(1 for i, v in enumerate(s) if v != 0 and v != GOAL[i])


# i es la posición actual y tile el valor de la ficha. GOAL.index busca su posición objetivo.
# En un tablero de ancho 3, i // 3 calcula la fila e i % 3 la columna.
def manhattan(i, tile):
    goal_i = GOAL.index(tile)
    # abs es una función incorporada: suma la diferencia de filas y la de columnas.
    return abs(i // 3 - goal_i // 3) + abs(i % 3 - goal_i % 3)


def h_manhattan(s):
    """Suma las distancias de las fichas. Es admisible y domina a h_misplaced."""
    # Suma la distancia de cada ficha al objetivo, sin contar el vacío.
    # Ignora obstáculos entre fichas: esa relajación explica por qué no sobreestima el costo real.
    return sum(manhattan(i, s[i]) for i in range(9) if s[i] != 0)


def h_bad(s):
    """NO ADMISIBLE: sobreestima hasta por un factor de 3. Observa las consecuencias."""
    # El factor 3 prioriza agresivamente lo que parece cercano al objetivo.
    # Puede ahorrar expansiones, pero deja de garantizar una solución de costo mínimo.
    return 3 * h_manhattan(s)


# Comprobación de dominancia: h_manhattan >= h_misplaced para todo estado, por construcción.
import random
# La semilla fija permite repetir el mismo recorrido pseudoaleatorio desde GOAL.
random.seed(0)
s = list(GOAL)
for _ in range(500):
    p = EightPuzzle(tuple(s))
    # Elige una acción legal y actualiza el tablero local: los estados siguen siendo alcanzables.
    # Se generan 500 pasos de un recorrido; no se exige que los 500 estados sean distintos.
    s = list(p.result(tuple(s), random.choice(p.actions(tuple(s)))))
    # assert lanza AssertionError si falla esta condición en el estado muestreado.
    # El muestreo comprueba ejemplos; la demostración general proviene de cómo se construyen las heurísticas.
    assert h_manhattan(tuple(s)) >= h_misplaced(tuple(s))
print("dominance h_manhattan >= h_misplaced verified on 500 random states")

dominance h_manhattan >= h_misplaced verified on 500 random states


## 4 · La tabla que resume toda la clase

Ejecuta las cuatro heurísticas sobre la misma instancia de profundidad 16.


In [4]:
# Ejecuta las cuatro heurísticas sobre el mismo problema y muestra sus medidas en una tabla.
# label es solo el texto que identifica la instancia en la salida.
def compare(start, label):
    # Crea una instancia del problema; cada llamada a astar crea su propia frontera y sus nodos.
    p = EightPuzzle(start)
    print(f"\n{label}   (true optimal = {TRUE[start]} moves)")
    print(f"  {'heuristic':<16}{'expansions':>12}{'sol.length':>12}{'optimal?':>10}{'time':>9}")
    print("  " + "-" * 60)
    # Las funciones son objetos: h guarda la referencia de una heurística, sin ejecutarla todavía.
    # name es la etiqueta de la fila y no cambia la búsqueda.
    for name, h in [("h_zero (=UCS)", h_zero), ("h_misplaced", h_misplaced),
                    ("h_manhattan", h_manhattan), ("h_bad (INADMIS.)", h_bad)]:
        # perf_counter permite medir tiempo transcurrido; el contador de expansiones mide trabajo algorítmico.
        t0 = time.perf_counter()
        # Desempaqueta el resultado: node es el objeto objetivo y exp la cantidad de expansiones.
        node, exp = astar(p, h)
        dt = time.perf_counter() - t0
        # node.path() invoca un método de instancia; len cuenta sus acciones.
        # En este dominio todos los pasos cuestan 1, así que longitud y costo coinciden.
        n = len(node.path())
        # Compara con la profundidad óptima conocida, no con una estimación de la propia heurística.
        ok = "yes" if n == TRUE[start] else "NO  <--"
        # Las f-strings alinean columnas: <16 a la izquierda, >12 a la derecha y .3f con tres decimales.
        print(f"  {name:<16}{exp:>12,}{n:>12}{ok:>10}{dt:>8.3f}s")


# Este diccionario relaciona cada tablero de referencia con su longitud óptima verificada.
TRUE = {D8: 8, D12: 12, D16: 16}
compare(D16, "depth-16 instance")


depth-16 instance   (true optimal = 16 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)         11,276          16       yes   0.080s
  h_misplaced              685          16       yes   0.004s
  h_manhattan              169          16       yes   0.001s
  h_bad (INADMIS.)          84          18   NO  <--   0.001s


### Las dos conclusiones

**1 · Manhattan domina a la heurística de fichas mal colocadas y reduce drásticamente las expansiones.**
La encuesta inicial se responde ahora con datos, en lugar de intuición. La dominancia no es
una preferencia estética: implica menos nodos, siempre y de forma demostrable.

**2 · ⚠️ `h_bad` es la más rápida y es incorrecta.**

Expande la menor cantidad de nodos *y* devuelve una solución subóptima. Dilo con claridad:

> **Siempre puedes ganar velocidad renunciando a la garantía; si no
> mides la calidad de la solución, no lo notarás.**

Esa oración conecta con todas las comparaciones con LLM de este curso. Un LLM
que busca caminos es `h_bad` con mejor publicidad: rápido, plausible y silenciosamente
subóptimo, a menos que lo compruebes.


## 5 · Escalabilidad — ¿Se mantiene el orden de desempeño?

Una instancia es una anécdota. Ejecuta las tres profundidades.


In [5]:
# Repite exactamente la comparación para tres dificultades y permite observar la escalabilidad.
# Conserva las etiquetas originales de salida para no alterar las cadenas del experimento.
for start, label in [(D8, "depth-8"), (D12, "depth-12"), (D16, "depth-16")]:
    compare(start, label + " instance")


depth-8 instance   (true optimal = 8 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)            172           8       yes   0.001s
  h_misplaced               16           8       yes   0.000s
  h_manhattan               11           8       yes   0.000s
  h_bad (INADMIS.)          11           8       yes   0.000s

depth-12 instance   (true optimal = 12 moves)
  heuristic         expansions  sol.length  optimal?     time
  ------------------------------------------------------------
  h_zero (=UCS)          2,177          12       yes   0.017s
  h_misplaced              110          12       yes   0.001s
  h_manhattan               28          12       yes   0.000s
  h_bad (INADMIS.)          35          12       yes   0.001s

depth-16 instance   (true optimal = 16 moves)
  heuristic         expansions  sol.length  optimal?     time
  ---------------------------------------------------

### ⚠️ Lee tus propios datos con honestidad

Observa lo que realmente ocurrió en las tres profundidades:

- En **profundidades 8 y 12**, `h_bad` encontró de todos modos el camino *óptimo*. Que sea
  no admisible significa que A\* **puede** devolver una respuesta subóptima, no que siempre
  lo haga. Un método sin garantías que acierta en instancias fáciles es el más
  peligroso, porque genera una falsa confianza.
- En **profundidad 12**, `h_bad` expandió *más* nodos que `h_manhattan` (35 frente a 28).
  Ni siquiera es más rápida de forma fiable. Sobreestimar distorsiona el orden de búsqueda: a veces
  ayuda y otras veces desperdicia trabajo.
- Solo en **profundidad 16** aparece el fallo: 84 expansiones y una respuesta de 18 movimientos
  cuando existe una de 16.

> **El fallo apareció únicamente en el mayor tamaño que probamos.** Si nos hubiéramos detenido en
> profundidad 12, habríamos concluido que `h_bad` funcionaba bien.
>
> Por eso, cada duelo de este curso exige una **gráfica de escalabilidad** en lugar de
> un único número de evaluación. Los fallos de garantía se ocultan en los tamaños pequeños.


## 6 · Factor de ramificación efectivo b\*

La forma honesta de comparar heurísticas entre distintos tamaños de problema. Si A\* expande `N`
nodos para encontrar una solución a profundidad `d`, entonces `b*` es el factor de ramificación
que necesitaría un árbol uniforme de profundidad `d` para contener `N` nodos.

**Un b\* cercano a 1 significa que la búsqueda fue casi directamente al objetivo.**


In [6]:
# Aproxima el factor de ramificación efectivo a partir de N expansiones y profundidad d.
# Busca b en el intervalo [lo, hi]; tol indica la anchura máxima deseada del intervalo final.
def effective_bf(N, d, lo=1.0000001, hi=10.0, tol=1e-6):
    """Resuelve N + 1 = 1 + b + b^2 + ... + b^d para b mediante bisección."""
    # Función interna, no método de una clase: conserva acceso al parámetro d de la función exterior.
    # Suma los nodos de los niveles 0 a d de un árbol uniforme con factor de ramificación b.
    def total(b):
        return sum(b ** i for i in range(d + 1))
    # Bisección: para b positivo la suma aumenta al crecer b, por lo que se puede reducir el intervalo a la mitad.
    # 200 limita el número de iteraciones incluso si no se alcanza antes la tolerancia.
    for _ in range(200):
        mid = (lo + hi) / 2
        # Si el punto medio produce pocos nodos, aumenta la cota inferior; si no, reduce la superior.
        if total(mid) < N + 1:
            lo = mid
        else:
            hi = mid
        # Se detiene por precisión del intervalo, no por una igualdad exacta de números reales.
        if hi - lo < tol:
            break
    # Devuelve el centro del intervalo final como aproximación de b.
    return (lo + hi) / 2


print(f"  {'heuristic':<16}{'depth':>7}{'expansions':>12}{'b*':>8}")
print("  " + "-" * 43)
# Usa solo las tres heurísticas admisibles para comparar la calidad manteniendo soluciones óptimas.
for start in (D8, D12, D16):
    for name, h in [("h_zero", h_zero), ("h_misplaced", h_misplaced),
                    ("h_manhattan", h_manhattan)]:
        node, exp = astar(EightPuzzle(start), h)
        # La profundidad se obtiene del plan encontrado; effective_bf usa exp como N y esta longitud como d.
        d = len(node.path())
        print(f"  {name:<16}{d:>7}{exp:>12,}{effective_bf(exp, d):>8.3f}")
    print()

print("""A better heuristic pushes b* toward 1.0 and KEEPS IT THERE as depth grows.
That stability is the property you actually want -- a heuristic that looks good on
easy instances and degrades on hard ones is the classic trap, and it is exactly
what the scaling plots in every Duel are designed to expose.""")

  heuristic         depth  expansions      b*
  -------------------------------------------
  h_zero                8         172   1.708
  h_misplaced           8          16   1.153
  h_manhattan           8          11   1.070

  h_zero               12       2,177   1.770
  h_misplaced          12         110   1.318
  h_manhattan          12          28   1.125

  h_zero               16      11,276   1.695
  h_misplaced          16         685   1.389
  h_manhattan          16         169   1.248

A better heuristic pushes b* toward 1.0 and KEEPS IT THERE as depth grows.
That stability is the property you actually want -- a heuristic that looks good on
easy instances and degrades on hard ones is the classic trap, and it is exactly
what the scaling plots in every Duel are designed to expose.


## 7 · De dónde vienen las heurísticas: problemas relajados

> **Toda heurística admisible es la solución exacta de un problema más sencillo.**

| Relajación | Heurística resultante |
|---|---|
| Una ficha puede moverse a cualquier lugar instantáneamente | `h_misplaced` |
| Una ficha puede moverse a cualquier casilla *adyacente*, aunque esté ocupada | `h_manhattan` |
| Sin relajación: el problema real | la propia respuesta |

Ambas heurísticas son *costos exactos* de un juego con reglas menos restrictivas. Por eso
nunca sobreestiman. Es una receta, no una conjetura: **elimina una restricción,
resuelve lo que queda y tendrás una heurística admisible.**

La semana 9 automatiza este proceso. Un planificador lee tu archivo de dominio, elimina los efectos
de borrado y deriva su propia heurística: el mismo principio, ejecutado por una máquina.


## 8 · Ejercicio

1. Añade la heurística de **conflicto lineal** y confirma que domina a Manhattan.
2. Construye `h_max = max(h_misplaced, h_manhattan)`. Demuestra que es admisible y después
   confirma que domina a ambas.
3. Define `h_bad = 1.1 * h_manhattan`. ¿Qué tan subóptimas resultan las soluciones? Esto es
   *A\* ponderada*, y la respuesta está acotada: encuentra la cota en AIMA §3.5.
4. Representa las expansiones frente a la profundidad para las cuatro heurísticas en las instancias
   de profundidad 8, 12 y 16. **Ese es el formato de gráfica que debes entregar en cada duelo de este
   curso.**
